In [ ]:
import re
import pandas as pd
import json
import openai
import time
import os
from openai import OpenAI

In [ ]:
with open('C:/Users/User/Desktop/chunk_split_list.json', 'r', encoding='utf-8') as f:
    chunk_wanted_list = json.load(f)
chunk_wanted_list

In [ ]:

# 🔑 OpenAI API 키
#client = OpenAI(
#  api_key=''
#)

# 🔁 함수: 채용공고 1개당 질문/답변 세트 생성
def generate_qa_pairs(job_info, n_pairs=1):
    prompt = f"""
당신은 채용 정보를 친절히 설명해주는 AI 비서입니다.

아래는 하나의 채용공고에서 추출된 여러 청크(부분 정보)입니다. 청크를 종합적으로 참고하여 채용 공고의 내용을 파악하고, 그 특징을 반영해주세요.

---
{job_info}
---

이 공고의 **특징을 반영하여**, 지원자가 실제로 궁금해할 만한 **다양하고 구체적인 질문을 {n_pairs}개** 만들어 주세요.
질문은 서로 **중복되지 않게** 하며, 너무 일반적인 질문(예: "근무지는 어디인가요?")은 피하고 **공고의 특성을 기반**으로 작성해주세요.

아래 형식으로 출력해주세요:

Q1: (질문 내용)
A1: (답변 내용)
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 채용공고를 보고 질문과 답변을 만드는 전문가야. 각 공고의 고유한 특징을 반영해 자연스러운 Q&A 세트를 만들어야 해."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
    )
    result = response.choices[0].message.content.strip()
    return result

In [ ]:
def toImproveText(text):
    if isinstance(text, list):  # 입력이 리스트인지 확인
        cleaned_list = []
        for item in text:
            cleaned_item = item.replace('\r', '').replace('\t', '')
            # +, #, :, %, ,는 유지하면서 나머지 특수문자 제거
            cleaned_item = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_item).strip()
            cleaned_list.append(cleaned_item)
        return cleaned_list
    elif isinstance(text, int):
        return text
    else:
        cleaned_text = text.replace('\r', '').replace('\t', '')
        cleaned_text = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_text).strip()
        return cleaned_text

In [ ]:
import random

random.shuffle(chunk_wanted_list)

chunk_wanted_list

In [ ]:
full_qa_list = []   # 전체 저장용
split_qa_list = []  # 2개마다 저장용
count = 1

for chunk_wanted_data_text in chunk_wanted_list:
    chunk_wanted_data_cleanedText = toImproveText(chunk_wanted_data_text)
    result = generate_qa_pairs(chunk_wanted_data_cleanedText)

    full_qa_list.append(result)     
    split_qa_list.append(result)    
    
    print(f"==================== {count}번째 완료 ====================")
    print(f"원문:\n{chunk_wanted_data_cleanedText}")
    print(f"{result}")

    if count == 1001: 
        break

    count += 1

In [ ]:
qa_list = []

for i in range(len(full_qa_list)):
    result = full_qa_list[i]
    chunk_text = chunk_wanted_list[i]

    qa_pairs = re.findall(r"Q\d+:\s*(.*?)\nA\d+:\s*(.*?)(?=\nQ\d+:|\Z)", result, re.DOTALL)

    for q, a in qa_pairs:
        qa_list.append({
            "instruction": q.strip(),
            "input": chunk_text.strip(),
            "output": a.strip()
        })

with open('chunk_qa_set_20250515_all.json', 'w', encoding='utf-8') as f:
    json.dump(qa_list, f, ensure_ascii=False, indent=2)

print("✅ 전체 JSON 저장 완료: chunk_qa_set_20250515_all.json")


In [ ]:
# csv 파일로 저장
csv_rows = []

for title, full_qa_list in qa_dict.items():
    for qa in full_qa_list:
        csv_rows.append({
            "질문": qa["질문"],
            "답변": qa["답변"]
        })

chunk_qa_set_df = pd.DataFrame(csv_rows)
chunk_qa_set_df.to_csv("chunk_qa_set_20250515_{count}.csv", index=False, encoding='utf-8-sig')
print("✅ 전체 csv 저장 완료: qa_set_20250515_all.csv")